<a href="https://colab.research.google.com/github/karye/Liu-labb1/blob/main/Lab_1_Tic_Tac_Toe_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Tic-Tac-Toe

In this laboration you will explore knowledge-based AI-agents for two-player-games. More specifically, how knowledge-based AI-agents can use reasoning and planning to play Tic-Tac-Toe.

The laboration first introduce the Tic-Tac-Toe game (as Python code), and the go through a number of different AI-agent types:
* Random Agent
* Rule-based Agent
* Planning Agent (Search)
* Adverserial Planning Agent (Minimax)
and invites you to implement your own rule-based agent.
As a bonus assignment, you are challenged (not required!) to implement a Adverserial Stochastic Planning Agent by implementing Monte Carlo Tree Search (MCTS).

**Objective:** Investigate limits, strengths and weaknesses of different AI-agent approaches and methods in the Tic-Tac-Toe domain.

**Do the labs top to bottom.**
<br />
**It is not needed to read further ahead than the current lab section.**

<br />
<hr />

### **Authors**
Mathias Ahlgren <br />
Mattias Tiger, mattias.tiger@liu.se

### **License**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

The lab (Notebook: code and instructions) is free to use, share, change and to make your own version suitable for your own students. The only restriction to this version and any new version is that 1) the same license (CC BY-NC-SA 4.0) is applied, 2) that the above mentioned authors are referred to as the authors of the original version and 3) that it cannot be used in a setting where a person or business has to pay to get access to, or in order to use, the version. I.e. full permission is given to use the lab, and subsequent versions, in schools or study circles, even running on paid-for computational resources (e.g. paid version of Google colab).
<hr />

# Tic-Tac-Toe - Lab 1.0 - The game
We'll start by setting up the board,  create the graphical interface and set up eventlisteners for the buttons.

### Step 1: Setup and Initializations

* **Task 1.0.1:** Make a shallow read-through of the code. Focus on the function names and descriptions.

  * Question 1: What is the difference between *check_winner(...)* and
  *check_tile_status(...)*?
    * check_tile-status() kollar om det går att göra ett drag.
  * Question 2: What determines the board size?
    * Konstanten BOARD_SIZE anger storleken, dvs 3x3

* **Task 1.0.2:** Run the code cell to initialize the game.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random

# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

def check_winner(board):
    """
    Checks if there is a winner on the board

    Args:
        board: Current game board (2D array).

    Returns:
      1 for player1
      2 for player2
      3 for Draw
      None for ongoing.
    """

    # Check rows and columns for a winner
    for i in range(BOARD_SIZE):
        if all(board[i][j] == board[i][0] for j in range(BOARD_SIZE)) and board[i][0] != 0:
            return board[i][0]
        if all(board[j][i] == board[0][i] for j in range(BOARD_SIZE)) and board[0][i] != 0:
            return board[0][i]

    # Check diagonals for a winner
    if all(board[k][k] == board[0][0] for k in range(BOARD_SIZE)) and board[0][0] != 0:
        return board[0][0]
    if all(board[k][BOARD_SIZE-1-k] == board[0][BOARD_SIZE-1] for k in range(BOARD_SIZE)) and board[0][BOARD_SIZE-1] != 0:
        return board[0][BOARD_SIZE-1]

    # Check for a draw (no winner and no empty spaces)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 3

    return None

def check_tile_status(board, row, col, player):
    """
    Simulates placing a move at the specified (row, col) and checks if it results in a win or draw.

    Args:
        board: Current game board (2D array).
        row: Row index of the tile to check.
        col: Column index of the tile to check.
        current_player: The player making the move (1 or 2).

    Returns:
        1 if Player 1 wins,
        2 if Player 2 wins,
        3 for a Draw,
        None for ongoing or if the tile is occupied.
    """

    if board[row][col] != 0:
        return None  # The tile is already filled, no result.

    # Temporarily place the move for current_player
    board[row][col] = player

    # Check row for a winner
    if all(board[row][j] == player for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Undo the move
        return player

    # Check column for a winner
    if all(board[j][col] == player for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Undo the move
        return player

    # Check main diagonal if applicable
    if row == col and all(board[k][k] == player for k in range(BOARD_SIZE)):
        board[row][col] = 0  # Undo the move
        return player

    # Check anti-diagonal if applicable
    if row + col == BOARD_SIZE - 1 and all(board[k][BOARD_SIZE-1-k] == player for k in range(BOARD_SIZE)):
        board[row][col] = 0  # Undo the move
        return player

    # Check for a draw (no winner and no empty spaces)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Undo the move
        return 3  # Draw

    board[row][col] = 0  # Undo the move if no win/draw
    return None

players = [
        ('Human', 'human')
    ]

# Dropdown menu for player1
player1_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Player1 (X)',
)

# Dropdown menu for player2
player2_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Player2 (0)',
)

# Initialize the global status label here
ai_status_label = widgets.Label(value="")

def create_buttons():
    """ Creates and returns a list of buttons """
    return [[widgets.Button(description='', layout=widgets.Layout(width='80px', height='80px')) for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

buttons = create_buttons()

def on_button_click(i, j):
    """ Event for making the move as a human and playing the Ai's next move """
    global current_player, board

    if board[i][j] == 0:  # If the cell is empty
        board[i][j] = current_player
        buttons[i][j].description = 'X' if current_player == 1 else 'O'
        buttons[i][j].disabled = True
        buttons[i][j].style.button_color = 'black'

        winner = check_winner(board)
        if winner:
            display_result(winner)
            return

        # Switch turns
        current_player = 1 if current_player == 2 else 2

        # If the current player is human we stop to wait for player to click a tile
        if get_current_player_type() == 'human':
            return

        # Stop and wait for the "step" button
        if player1_dropdown.value != 'human' and player2_dropdown.value != 'human':
            return

        # If it's an AI move, make the move immediately
        if get_current_player_type() != 'human':
            ai_move = get_ai_move(get_current_player_dropdown())
            if ai_move:
                on_button_click(ai_move[0], ai_move[1])

def get_ai_move(player_dropdown):
    """
    Gets the AI's next move based on the chosen strategy.
    """
    global ai_status_label, current_player, board

    selected_strategy = player_dropdown.value
    ai_status_label.value = "AI is thinking..."

    ai_move = None

    if selected_strategy == 'random':
        ai_move = find_best_move_random(board)
    elif selected_strategy == 'my_ai_strategy':
        ai_move = find_best_move(board, current_player)
    elif selected_strategy == 'rule_based':
        ai_move = find_best_move_rule_based(board, current_player)
    elif selected_strategy == 'minimax':
        ai_move = find_best_move_min_max(board, current_player, use_alpha_beta=True)
    elif selected_strategy == 'dfs':
        ai_move = find_best_move_dfs(board, current_player)
    elif selected_strategy == 'bfs':
        ai_move = find_best_move_bfs(board, current_player)

    ai_status_label.value = "AI is done!"
    return ai_move

def get_current_player_type():
    """
    Determines if the current player is AI or human.
    """
    return player1_dropdown.value if current_player == 1 else player2_dropdown.value

def get_current_player_dropdown():
    """
    Gets the appropriate dropdown for the current player (Player 1 or Player 2).
    """
    return player1_dropdown if current_player == 1 else player2_dropdown

def display_result(winner):
    """
    Displays the result of the game.
    """
    result = ""
    if winner == 1:
        result = "Player 1 wins" if player1_dropdown.value == 'human' else "AI (Player 1) wins"
    elif winner == 2:
        result = "Player 2 wins" if player2_dropdown.value == 'human' else "AI (Player 2) wins"
    elif winner == 3:
        result = "It's a draw"

    result_label.value = result
    disable_all_buttons()

def disable_all_buttons():
    """
    Disables all buttons.
    """
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].disabled = True

def step_game(_=None):
    global current_player
    """
    Function to step through the AI vs AI moves when the "Step" button is pressed.
    """
    if player1_dropdown.value != 'human' and player2_dropdown.value != 'human': # If Ai vs Ai we diable buttons
        disable_all_buttons()

    if get_current_player_type() != 'human':
        ai_move = get_ai_move(get_current_player_dropdown())
        if ai_move:
            on_button_click(ai_move[0], ai_move[1])

def new_game(_=None):
    """
    Starts and draws the new game.
    """
    global board, current_player, buttons, result_label, ai_status_label
    clear_output()

    board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]
    current_player = 1

    buttons = create_buttons()

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].on_click(lambda btn, i=i, j=j: on_button_click(i, j))

    grid = widgets.GridBox(children=[button for row in buttons for button in row],
                          layout=widgets.Layout(grid_template_columns=f"repeat({BOARD_SIZE}, 90px)"))

    result_label = widgets.Label(value="") #
    ai_status_label.value = ""

    new_game_button = widgets.Button(description="New game", layout=widgets.Layout(width='200px', height='40px'))
    step_button = widgets.Button(description="Step", layout=widgets.Layout(width='200px', height='40px'))

    new_game_button.on_click(new_game)
    step_button.on_click(step_game)

    buttons_box = widgets.HBox([new_game_button, step_button], layout=widgets.Layout(justify_content='flex-start'))

    display(player1_dropdown)
    display(player2_dropdown)
    display(grid)
    display(result_label)
    display(ai_status_label)
    display(buttons_box)

### Step 2: Creating the graphical interface

(Run the code-cell)

* **Task 1.0.3:** Play the game against yourself.
  * Question 1: Does the game seem simple to win?
    * Ja
  * Question 2: If you play the game perfectly as both players, do you believe that any player will ever win the game?
    * Nej
  * Question 3: Do you believe that the location of the first move affect if a game is won or lost, assuming perfect players?
    * Ja

In [ ]:
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

player1_dropdown.options = [
      ('Human', 'human')
  ]
player2_dropdown.options = [
      ('Human', 'human')
  ]

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.1 - Random Agent


* **Task 1.1.1:** Read the code.
  * Question 1: What does *random.choice(available_moves)* do?
  * random.choice() slumpar ut ett element ur listan
* **Task 1.1.2:** Run the code cells and play against the Random Agent.
  * Question 2: Is it easy for you to win against it?
  * Ja
  * Question 3: Is the difficulty of you winning against it affected by if you or Random Agent get to make the first move?
  * Nej
* **Task 1.1.3:** Let the Random Agent play against itself.
  * Question 4: Does the gameplay seem intelligent?
  * Question 5: What would be the behavior of a more intelligent agent?
  * Question 6: What would be some simple improvements (conceptually) that would make the agent better?
  
(If the AI-agent is supposed to make the first move, press **step**.)

In [ ]:
def find_best_move_random(board):
    """
    Example strategy of a random strategy.
    Current player is not used (a dummy input).
    """

    # Get the list of all empty positions on the board
    available_moves = [(i, j) for i in range(BOARD_SIZE) for j in range(BOARD_SIZE) if board[i][j] == 0]

    if available_moves:
        return random.choice(available_moves)

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Human', 'human'),
    ('Random', 'random')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.2 - Your own Agent

**Will you be able beat your own agent?**

* **Task 1.2.1:** Read the code.
  * Question 1: What does *Rule 1* do and how does it work?
* **Task 1.2.2:** Extend the code to make your own AI agent play the game.<br/>
  (Minimal requirement: If the board size is 3 and the board is empty, make the first mark at the center of the baord.)
  * Question 1: What are some (1-3) good rules?
  * Question 2: Which rules did you implement?
* **Task 1.2.3:** Run the code cells and play against the your own agent.
  * Question 2: Is it easy for you to win against it?
  * Question 3: Is the difficulty of you winning against it affected by if you or your own agent get to make the first move?
* **Task 1.2.4:** Let your own agent play against itself.
  * Question 4: Does the gameplay seem intelligent?
  * Question 5: What would be the behavior of a more intelligent agent?
  * Question 6: What would be some simple improvements (conceptually) that would make the agent better?

  

In [ ]:
def find_best_move(board, current_player):
    """
    Function for the students to modify

    The goal is to implement rules for how AI will play.
    You have already been given the first and last rules.

    Return a tuple of (i, j) that will be the Ai's move. (0, 0) is top left
    """

    # Rule 1: Win if possible
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Winning move found

    # TODO: Add rules
    # ...............

    # Last rule any move
    return find_best_move_random(board)

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.3 - Rule-based Agent

* **Task 1.3.1:** Read the code.
  * Question 1: Explain the strategy of the Rule-based Agent in your own words.
  * Question 2: What is different in terms of *thinking ahead* between *Rule 1* and *Rule 2*?
  * QUestion 3: If you have to remove two rules, which do you think would be the least bad to remove (degrade gameplay performance the least)?
* **Task 1.3.2:** Run the code cells and play against the Rule-based Agent.
  * Question 4: Is it easy for you to win against it?
  * Question 5: Is the difficulty of you winning against it affected by if you or the Rule-based Agent get to make the first move?
* **Task 1.3.3:** Let the  Rule-based Agent play against the random agent.
  * Question 6: Which agent seems more intelligent?
  * Question 7: Does it matter which agent that starts for how the game ends up?
* **Task 1.3.4:** Let your the Rule-based Agent play against itself.
  * Question 8: Does the gameplay seem intelligent?
  * Question 9: What would be the behavior of a more intelligent agent?
  * Question 10: What would be some simple improvements (conceptually) that would make the agent better?


In [ ]:
def find_best_move_rule_based(board, current_player):
    """
    Example function of a rule-based strategy, using check_tile_status.
    """
    opponent = 2 if current_player == 1 else 1

    # Rule 1: Win if possible
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Winning move found

    # Rule 2: Block opponent's win
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, opponent) == opponent:
                    return (i, j)  # Blocking move found

    # Rule 3: Take the center if available (only for 3x3 board currently)
    if BOARD_SIZE == 3 and board[1][1] == 0:
        return (1, 1)

    # Rule 4: Take one of the corners if available
    for i, j in [(0, 0), (0, BOARD_SIZE - 1), (BOARD_SIZE - 1, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]:
        if board[i][j] == 0:
            return (i, j)

    # Rule 5: Take any available space
    return find_best_move_random(board)

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.4 - Planning Agent (Search)

* **Task 1.4.1:** Read the code.
  * Question 1: Explain the strategies of the search-based agents in your own words.

* **Task 1.4.2:** Run the code cells and play against the search-based Agent.
  * Question 2: Is it easy for you to win against them?
  * Question 3: Why are search based agents that only look for winning moves not suited for adversarial games?
  * Question 4: How do you think a search based strategy could be improved to play better?



In [ ]:

from collections import deque

def find_best_move_dfs(board, player, max_depth=6):
    """
    Finds the best move for the AI using Depth-First Search (DFS).
    This function searches the game tree recursively up to a certain depth
    and selects the first move that leads to a win.
    If no winning path is found within the depth limit, it selects a random move.
    """
    opponent = 2 if player == 1 else 1

    def dfs(board, current_player, depth):
        winner = check_winner(board)
        if winner == player:
            return True  # Winning path found
        elif winner == opponent or winner == 3:
            return False  # Opponent wins or draw
        elif depth == max_depth:
            return False  # Max depth reached without finding a win

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    # Make a move
                    board[i][j] = current_player
                    found_win = dfs(board, 2 if current_player == 1 else 1, depth + 1)
                    # Undo the move
                    board[i][j] = 0

                    if current_player == player and found_win:
                        return True  # Winning path found
        return False  # No winning path found at this branch

    # Try all possible moves
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                # Make a move
                board[i][j] = player
                found_win = dfs(board, opponent, 1)
                # Undo the move
                board[i][j] = 0

                if found_win:
                    return (i, j)  # Return the first move leading to a win

    # No winning path found, select a random move
    return find_best_move_random(board)

def find_best_move_bfs(board, current_player, max_depth = 5):
    """
    Finds the best move for the AI using Breadth-First Search (BFS).
    This function explores the game tree level by level to find the quickest winning move.
    """
    opponent = 2 if current_player == 1 else 1
    queue = deque()

    # Initialize the queue with all possible moves for the current player
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                new_board = [row[:] for row in board]
                new_board[i][j] = current_player
                queue.append((new_board, (i, j), current_player, 1))  # (board state, move, player, depth)

    best_move = None
    min_depth = float('inf')

    while queue:
        current_board, move, player, depth = queue.popleft()
        winner = check_winner(current_board)

        if winner == current_player:
            # Found a winning move
            if depth < min_depth:
                min_depth = depth
                best_move = move
            continue
        elif winner == opponent or winner == 3:
            continue # skip draws or opponent win

        if depth >= max_depth:
            continue # Stop on max_depth

        # Generate possible moves for the next player
        next_player = opponent if player == current_player else current_player

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if current_board[i][j] == 0:
                    new_board = [row[:] for row in current_board]
                    new_board[i][j] = next_player
                    queue.append((new_board, move, next_player, depth + 1))

    if best_move is not None:
        return best_move
    else:
        # If no winning move found, fallback to a random move
        return find_best_move_random(board)



In [ ]:
# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [

    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based'),
    ('Depth first search', 'dfs'),
    ('Breadth first search', 'bfs')
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'dfs'

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.5 - Adverserial Planning Agent (Minimax)

* **Task 1.5.1:** Read the code.
  * Question 1: What are the different possible return values of the *minimax(...)* function and what does each of them mean?
  * Question 2: How many times is the function *minimax(...)* called by the function *find_best_move_min_max(...)*?
  * Question 3: Do you think that *minimax(...)* is called more times from *minimax(...)* itself than from *find_best_move_min_max(...)*?
  * Question 4: Do you believe that this agent is faster, slower or similar in terms of computation time to run?
* **Task 1.5.2:** Run the code cells and play against the Adverserial Planning Agent.
  * Question 5: Is it easy for you to win against it?
  * Question 6: Is the difficulty of you winning against it affected by if you or the agent get to make the first move?
* **Task 1.5.3:** Let the Adverserial Planning Agent (Minimax) play against the other agents.
  * Question 7: Which agent seems more intelligent?
  * Question 8: Does it matter which agent that starts for how the game ends up?
  * Question 9: Does it seem to matter if you increase the board size to 4x4?
  * Question 10: How does the max_depth parameter affect the minimax algorithm?

* **Task 1.5.4:** Let the Agent play against itself.
  * Question 10: Does the gameplay seem intelligent?

In [ ]:
def evaluate_board(board, current_player):
    opponent = 2 if current_player == 1 else 1
    score = 0

    center = (BOARD_SIZE - 1) // 2

    # Positional scores
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == current_player:
                # Center position
                if i == center and j == center:
                    score += 3  # Assign higher value to center
                # Corner positions
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score += 2  # Assign medium value to corners
                else:
                    score += 1  # Assign lower value to edges
            elif board[i][j] == opponent:
                # Subtract positional scores for opponent's pieces
                if i == center and j == center:
                    score -= 3
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score -= 2
                else:
                    score -= 1

    # Evaluate potential winning lines for both players
    lines = []

    # Rows and columns
    for i in range(BOARD_SIZE):
        lines.append(board[i])    # Row
        lines.append([board[j][i] for j in range(BOARD_SIZE)])    # Column

    # Diagonals
    lines.append([board[k][k] for k in range(BOARD_SIZE)])
    lines.append([board[k][BOARD_SIZE-1-k] for k in range(BOARD_SIZE)])

    for line in lines:
        score += evaluate_line(line, current_player, opponent)

    return score


def evaluate_line(line, player, opponent):
    score = 0
    player_count = sum(1 for x in line if x == player)
    opponent_count = sum(1 for x in line if x == opponent)
    empty_count = sum(1 for x in line if x == 0)

    if player_count == 2 and empty_count == 1:
        score += 10  # Potential winning line for AI
    elif player_count == 1 and empty_count == 2:
        score += 1   # Weak potential line for AI
    elif opponent_count == 2 and empty_count == 1:
        score -= 9   # Potential winning line for opponent (block it)
    elif opponent_count == 1 and empty_count == 2:
        score -= 1   # Weak potential line for opponent

    return score

def minimax(board, depth, is_maximizing, max_depth, current_player, alpha=float('-inf'), beta=float('inf'), use_alpha_beta=False):
    opponent = 2 if current_player == 1 else 1

    winner = check_winner(board)
    if winner == current_player:
        return 100 - depth  # Prefer quicker wins
    elif winner == opponent:
        return -100 + depth  # Prefer longer losses
    elif winner == 3:
        return 0  # Draw
    elif depth == max_depth:
        # Use the evaluation function
        return evaluate_board(board, current_player)

    if is_maximizing:
        best_score = float('-inf')
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    board[i][j] = current_player
                    score = minimax(board, depth + 1, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i][j] = 0
                    best_score = max(score, best_score)
                    if use_alpha_beta:
                        alpha = max(alpha, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score
    else:
        best_score = float('inf')
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    board[i][j] = opponent
                    score = minimax(board, depth + 1, True, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i][j] = 0
                    best_score = min(score, best_score)
                    if use_alpha_beta:
                        beta = min(beta, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score

def find_best_move_min_max(board, current_player, use_alpha_beta=True, max_depth=8):
    """
    Find the best move using the Minimax algorithm with optional Alpha-Beta pruning.
    The AI evaluates moves by maximizing its own potential and minimizing the opponent's potential.
    """
    best_move = None
    best_score = float('-inf')

    if BOARD_SIZE >= 4:
      max_depth = 5

    opponent = 2 if current_player == 1 else 1

    # Initialize alpha and beta values for alpha-beta pruning
    alpha = float('-inf')
    beta = float('inf')

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                # Make the move
                board[i][j] = current_player
                # Call minimax recursively and choose the maximum value
                score = minimax(board, 0, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                # Undo the move
                board[i][j] = 0

                if score > best_score:
                    best_score = score
                    best_move = (i, j)

                # Update alpha value if alpha-beta pruning is enabled
                if use_alpha_beta:
                    alpha = max(alpha, best_score)
                    if beta <= alpha:
                        break  # Alpha cutoff
        if use_alpha_beta and beta <= alpha:
            break  # Alpha cutoff

    return best_move

In [ ]:
# Board configuration
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Human', 'human'),
    ('Random', 'random'),
    ('My AI Strategy', 'my_ai_strategy'),
    ('Rule-based', 'rule_based'),
    ('Depth first search', 'dfs'),
    ('Breadth first search', 'bfs'),
    ('MinMax', 'minimax'),
]

# Dropdown menu for player1 and player2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'minimax'

# Start the game
new_game()

# Tic-Tac-Toe - Lab 1.6 - Adverserial Stochastic Planning Agent (MCTS)

**(Optional)**

* **Task 1.6.1:** Implement MCTS
  * Question 1: Which source did you use?
  * Question 2: What differences with Minimax do you expect?
* **Task 1.6.2:** Run the code cells and play against the Adverserial Stochastic Planning Agent (MCTS).
  * Question 5: Is it easy for you to win against it?
  * Question 6: Is the difficulty of you winning against it affected by if you or the agent get to make the first move?
* **Task 1.5.3:** Let the Adverserial Stochastic Planning Agent (MCTS) play against the other agents.
  * Question 7: Which agent seems more intelligent?
  * Question 8: Does it matter which agent that starts for how the game ends up?
  * Question 9: Does it matter if you increase the board size to 4x4?
* **Task 1.5.4:** Let the Adverserial Stochastic Planning Agent (MCTS) play against itself.
  * Question 10: Does the gameplay seem intelligent?